<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [2]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [3]:
import folium
import pandas as pd

In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [5]:

import folium

# Create a global map
spacex_map = folium.Map(
    location=[25, -40],
    zoom_start=4
)

# Launch site coordinates from the IBM dataset
launch_sites = {
    "CCAFS LC-40": [28.562302, -80.577356],
    "CCAFS SLC-40": [28.563197, -80.576820],
    "KSC LC-39A": [28.573255, -80.646895],
    "VAFB SLC-4E": [34.632834, -120.610745]
}

# Assign different colors to launch sites
colors = ["red", "blue", "green", "purple"]

# Add markers for all launch sites
for (site, coordinates), color in zip(
    launch_sites.items(), colors
):
    folium.Marker(
        location=coordinates,
        popup=site,
        tooltip=site,
        icon=folium.Icon(color=color)
    ).add_to(spacex_map)

# Display the map
spacex_map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [6]:

# Download and read the spacex_launch_geo.csv dataset

import pandas as pd
from js import fetch
import io

# Dataset URL
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'

# Download the CSV file
resp = await fetch(URL)

# Convert the response into a file-like object
spacex_csv_file = io.BytesIO(
    (await resp.arrayBuffer()).to_py()
)

# Read the CSV file into a Pandas DataFrame
spacex_df = pd.read_csv(spacex_csv_file)

# Display the first five rows
spacex_df.head()


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [7]:

# Select relevant columns from the dataset
# Launch Site, Lat (Latitude), Long (Longitude), and class

spacex_df = spacex_df[
    ['Launch Site', 'Lat', 'Long', 'class']
]

# Group the data by launch site and select the first record
launch_sites_df = spacex_df.groupby(
    ['Launch Site'],
    as_index=False
).first()

# Keep only launch site names and coordinates
launch_sites_df = launch_sites_df[
    ['Launch Site', 'Lat', 'Long']
]

# Display the launch site coordinates
launch_sites_df


,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [8]:
# Start location is NASA Johnson Space Center
import folium

# Define the coordinates of NASA Johnson Space Center
# Latitude: 29.559684888503615
# Longitude: -95.0830971930759
nasa_coordinate = [
    29.559684888503615,
    -95.0830971930759
]

# Create a Folium map centered at NASA Johnson Space Center
site_map = folium.Map(
    location=nasa_coordinate,
    zoom_start=10
)

# Display the map
site_map


We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [9]:

import folium
from folium.features import DivIcon

# Recreate the map centered at NASA Johnson Space Center
site_map = folium.Map(
    location=nasa_coordinate,
    zoom_start=15
)

# Add a blue circle around NASA Johnson Space Center
circle = folium.Circle(
    location=nasa_coordinate,
    radius=50,
    color='blue',
    fill=True,
    fill_color='blue',
    fill_opacity=0.3,
    popup='NASA Johnson Space Center'
)

# Add the circle to the map
circle.add_to(site_map)

# Add a blue text label
marker = folium.Marker(
    location=nasa_coordinate,
    icon=DivIcon(
        icon_size=(150, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size:12px; color:blue;"><b>NASA JSC</b></div>'
    )
)

# Add the text marker to the map
marker.add_to(site_map)

# Display the map
site_map


and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [10]:

import folium
from folium.features import DivIcon

# Create a map centered over the United States
site_map = folium.Map(
    location=[32, -100],
    zoom_start=4
)

# Iterate through each launch site
for index, row in launch_sites_df.iterrows():

    # Get latitude and longitude from the dataset
    coordinate = [row['Lat'], row['Long']]

    # Add a circle for each launch site
    folium.Circle(
        location=coordinate,
        radius=1000,
        color='#000000',
        fill=True,
        fill_opacity=0.3
    ).add_child(
        folium.Popup(row['Launch Site'])
    ).add_to(site_map)

    # Add the launch site name as a text label
    folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(150, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color: #d35400;"><b>%s</b></div>'
            % row['Launch Site']
        )
    ).add_to(site_map)

# Display the map
site_map


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


Are all launch sites in proximity to the Equator line?

No. All four launch sites are located in the Northern Hemisphere, but they are not very close to the Equator.
- The three Florida launch sites are approximately 28.5° North latitude.
- VAFB SLC-4E in California is approximately 34.6° North latitude.
- The Equator is at 0° latitude.
- 
Finding: The Florida launch sites are closer to the Equator than the California launch site. However, none of the four sites is located directly near the Equator.

Are all launch sites in very close proximity to the coast?

Yes. All four launch sites are situated near coastlines.
- CCAFS LC-40: Near the Atlantic coast in Florida.
- CCAFS SLC-40: Near the Atlantic coast in Florida.
- KSC LC-39A: Located near the Atlantic coast in Florida.
- VAFB SLC-4E: Located near the Pacific coast in California.
- 
Finding: All four launch sites have coastal locations, which are advantageous for rocket launches because they provide open flight paths over the ocean and help reduce risks to populated areas.

Conclusion:
SpaceX launch sites are strategically positioned near coastlines to support safe rocket launches and provide open flight paths over the ocean. Although none is particularly close to the Equator, the Florida locations offer a more favorable latitude for launches that benefit from Earth's rotational velocity.

In [22]:
# Display all column names in the dataset
print(spacex_df.columns.tolist())

['Launch Site', 'Lat', 'Long', 'class', 'marker_color']


In [23]:


# Add markers for successful and failed launches
for index, row in spacex_df.iterrows():

    # Get latitude and longitude from the dataset
    coordinate = [row['Lat'], row['Long']]

    # Assign marker colors based on launch outcome
    if row['class'] == 1:
        color = 'green'  # Successful launch
    else:
        color = 'red'    # Failed launch

    # Add a circle marker for each launch
    folium.CircleMarker(
        location=coordinate,
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"Class: {row['class']}"
    ).add_to(site_map)

# Display the map
site_map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [24]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [25]:

# Import MarkerCluster
from folium.plugins import MarkerCluster

# Create a marker cluster
marker_cluster = MarkerCluster()

*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [26]:

# Assign marker colors based on the launch outcome
# Class 1 = Successful launch (Green)
# Class 0 = Failed launch (Red)

spacex_df['marker_color'] = spacex_df['class'].apply(
    lambda x: 'green' if x == 1 else 'red'
)

# Display the first five rows
spacex_df.head()


,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [27]:

# Add the marker cluster to the current site map
site_map.add_child(marker_cluster)

# Add a marker for each launch result
for index, record in spacex_df.iterrows():

    # Get launch coordinates (Latitude, Longitude)
    coordinate = [record['Lat'], record['Long']]

    # Create a marker for each launch
    marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(
            color='white',
            icon_color=record['marker_color'],
            icon='info-sign'
        )
    )

    # Add the marker to the cluster
    marker_cluster.add_child(marker)

# Display the map
site_map


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [28]:
# Display launch site coordinates
launch_sites

{'CCAFS LC-40': [28.562302, -80.577356],
 'CCAFS SLC-40': [28.563197, -80.57682],
 'KSC LC-39A': [28.573255, -80.646895],
 'VAFB SLC-4E': [34.632834, -120.610745]}

In [29]:

import math

# Define a function to calculate the distance between two coordinates
def calculate_distance(lat1, lon1, lat2, lon2):

    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(
        math.radians,
        [lat1, lon1, lat2, lon2]
    )

    # Calculate the differences in latitude and longitude
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # Apply the Haversine formula
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2)
        * math.sin(dlon / 2) ** 2
    )

    # Calculate the central angle
    c = 2 * math.asin(math.sqrt(a))

    # Calculate distance using Earth's radius (6,371 km)
    distance = 6371 * c

    return distance


Next, we need to explore and analyze the proximities of launch sites.


In [30]:
# Calculate distance between launch site and coastline
distance = calculate_distance(
    28.562302, -80.577356,
    28.56340, -80.56796
)

print("Distance to nearest coastline:", round(distance, 3), "km")


Distance to nearest coastline: 0.926 km


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [31]:


# Add Mouse Position to display coordinates on the map
from folium.plugins import MousePosition

formatter = "function(num) {return L.Util.formatNum(num, 5);};"

mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=5,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter
)

site_map.add_child(mouse_position)

# Display map
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [32]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # Approximate radius of Earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        sin(dlat / 2)**2
        + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance


# CCAFS LC-40 launch site coordinates
# Nearest coastline coordinates
distance = calculate_distance(
    28.562302, -80.577356,
    28.56340, -80.56796
)

print("Distance to nearest coastline:", round(distance, 3), "km")

Distance to nearest coastline: 0.926 km


*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [33]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
from folium.plugins import MousePosition

# Display latitude and longitude when you move your mouse over the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"

MousePosition(
    position='topright',
    separator=' | ',
    prefix='Lat/Lon:',
    lat_formatter=formatter,
    lng_formatter=formatter
).add_to(site_map)

site_map

In [34]:
# Launch site coordinates (example: CCAFS SLC-40)
launch_site_lat = 28.563197
launch_site_lon = -80.576820
	

# Closest coastline coordinates (replace with your observed values)
coastline_lat = 28.5679
coastline_lon = -80.56695

# Calculate the distance in kilometers
distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon
)

print("Distance to the closest coastline:", distance_coastline, "km")

Distance to the closest coastline: 1.096963278002838 km


In [35]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )
from folium.features import DivIcon

# Closest coastline coordinates
coastline = [coastline_lat, coastline_lon]

# Create a marker displaying the distance
distance_marker = folium.Marker(
    coastline,
    icon=DivIcon(
        icon_size=(200, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_coastline)
    )
)

# Add the marker to the map
site_map.add_child(distance_marker)

# Display the map
site_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [36]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# lines=folium.PolyLine(locations=coordinates, weight=1)
# Coordinates of the launch site and coastline point
coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]

# Create a PolyLine between the two points
lines = folium.PolyLine(
    locations=coordinates,
    weight=1,
    color='blue'
)

# Add the line to the map
site_map.add_child(lines)

# Display the map
site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [32]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
# Coordinates of the launch site
launch_site_lat = 28.563197
launch_site_lon = -80.577356

# Replace these with the coordinates of the closest city, railway, or highway
closest_lat = 28.55
closest_lon = -80.80

# Calculate the distance in kilometers
distance = calculate_distance(
    launch_site_lat, launch_site_lon,
    closest_lat, closest_lon
)

# Create a marker displaying the distance
from folium.features import DivIcon

distance_marker = folium.Marker(
    [closest_lat, closest_lon],
    icon=DivIcon(
        icon_size=(200, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color: #d35400;"><b>{distance:.2f} KM</b></div>'
    )
)

# Add the marker to the map
site_map.add_child(distance_marker)

# Draw a line between the launch site and the nearby location
coordinates = [
    [launch_site_lat, launch_site_lon],
    [closest_lat, closest_lon]
]

lines = folium.PolyLine(
    locations=coordinates,
    weight=2,
    color='blue'
)

site_map.add_child(lines)

# Display the map
site_map


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


Based on the geographical locations of the four SpaceX launch sites and the proximity analysis using Folium maps, we can make the following observations.

1. Are launch sites in close proximity to railways?
-Yes. Launch sites generally have access to railway infrastructure or transportation networks that can support the movement of heavy rocket components and equipment. However, the exact distance to the nearest railway needs to be verified using the map coordinates.

2. Are launch sites in close proximity to highways?
-Yes. Launch facilities have access to nearby road and highway networks. These are important for transporting rocket components, fuel, equipment, and personnel.


3. Are launch sites in close proximity to coastlines?
Yes. This is one of the most noticeable geographical characteristics.
- CCAFS LC-40 and CCAFS SLC-40 are located near the Atlantic coast in Florida.
- KSC LC-39A is also located near the Atlantic coast.
- VAFB SLC-4E is located near the Pacific coast in California.
Coastal locations provide safer launch corridors over the ocean, reducing the risk of rocket debris or launch failures affecting populated areas.

4. Do launch sites keep a certain distance away from cities?
Yes. Launch facilities are generally located away from densely populated urban areas.

This provides:
- Improved public safety during launches.
- Reduced risks to nearby residents.
- Space for launch infrastructure and safety zones.
- Better control over restricted airspace and surrounding areas.

Overall conclusion
The geographical analysis shows that SpaceX launch sites are strategically located to balance transportation accessibility, public safety, and launch requirements. Their proximity to coastlines allows rockets to launch over the ocean, while access to highways and transportation infrastructure supports the movement of equipment and personnel. At the same time, maintaining separation from densely populated cities helps minimize potential risks.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
